# Pinning OpenVINO policy output with a seed — PoC

Reference for the **golden-action** tier in
[`docs/design/openvino-validation.md`](../docs/design/openvino-validation.md).
It demonstrates two claims on a real exported policy (pi05), end to end:

1. **The output is non-deterministic** because the exported IR draws its
   denoising noise from a single `RandomUniform` op shipped with
   `global_seed=0, op_seed=0` — which the OpenVINO spec defines as a
   non-deterministic sequence.
2. **Seeding fixes the noise _sequence_, not a single value.** Each call advances
   the sequence, but the Nth call after a fresh load is always the same. So
   golden-action records the sequence once and replays it against a freshly
   loaded model.

Seeding is done as an **in-memory graph transform** — read the model, rewrite
every `RandomUniform` node with a non-zero seed, then compile. That is the
approach the real implementation should use. This notebook is a PoC, not
production code; determinism holds per `(device, precision, OpenVINO version)`.


In [ ]:
from pathlib import Path

import numpy as np
import openvino as ov
from openvino import opset8
from openvino.utils import replace_node

MODEL = Path.home() / "models" / "pi05_cups_rtc" / "pi05.xml"
DEVICE = "CPU"
SEED = 42
CALLS = 10

core = ov.Core()


_NP_DTYPE = {
    "f32": np.float32,
    "f16": np.float16,
    "f64": np.float64,
    "i64": np.int64,
    "i32": np.int32,
    "u8": np.uint8,
    "boolean": np.bool_,
}


def build_fixed_inputs(compiled):
    """Deterministic, well-typed inputs for every port (identical on every call),
    so the only possible source of output variation is the in-graph RandomUniform.
    """
    out = {}
    for port in compiled.inputs:
        dtype = _NP_DTYPE[port.get_element_type().get_type_name()]
        shape = tuple(port.get_shape())
        out[port.get_any_name()] = np.ones(shape, bool) if dtype is np.bool_ else np.zeros(shape, dtype)
    return out


def action_output_name(compiled):
    names = [p.get_any_name() for p in compiled.outputs]
    return "action" if "action" in names else names[0]


def load(seed=None):
    """Read the IR, optionally seed every RandomUniform node, and compile a fresh model.
    Returns (compiled_model, number_of_RandomUniform_ops).
    """
    model = core.read_model(str(MODEL))
    rng_ops = [op for op in model.get_ops() if op.get_type_name() == "RandomUniform"]
    if seed is not None:
        for op in rng_ops:
            seeded = opset8.random_uniform(
                op.input_value(0),
                op.input_value(1),
                op.input_value(2),
                op.get_output_element_type(0).get_type_name(),
                global_seed=seed,
                op_seed=seed,
            )
            seeded.set_friendly_name(op.get_friendly_name())
            replace_node(op, seeded)
    return core.compile_model(model, DEVICE), len(rng_ops)


def run(compiled, inputs, out_name, calls):
    """Run `calls` inferences with identical input; return the action chunk each call."""
    out = compiled.output(out_name)
    return [np.array(compiled(inputs)[out]) for _ in range(calls)]


def max_abs(a, b):
    return float(np.abs(a.astype(np.float64) - b.astype(np.float64)).max())

## Claim 1 — the stock model is non-deterministic

Load the model **unseeded** (the `RandomUniform` op keeps its exported
`global_seed=0, op_seed=0`) and run twice with an identical observation. The two
action chunks differ.


In [ ]:
# Baseline: unseeded model, two calls with identical input -> outputs differ.
base, n_rng = load(seed=None)
inputs = build_fixed_inputs(base)
out_name = action_output_name(base)

a, b = run(base, inputs, out_name, 2)

## The fix — seed the `RandomUniform` op, then record the sequence

`load(seed=SEED)` rewrites the op with non-zero seeds before compiling. We record
the first `CALLS` action chunks. Note the calls still differ from each other: a
seed fixes the *sequence*, and each inference draws the next value in it.


In [ ]:
# Record: seed + compile, then capture the first CALLS action chunks.
rec_model, _ = load(seed=SEED)
recorded = run(rec_model, build_fixed_inputs(rec_model), out_name, CALLS)

intra = max(max_abs(recorded[k], recorded[0]) for k in range(1, CALLS))

## Claim 2 — the seeded sequence reproduces across a reload

Throw the model away, **reload and reseed from scratch**, replay the same `CALLS`
inferences, and compare call by call. Every chunk matches the recorded reference
bit-for-bit — so golden-action can use plain equality (no tolerance) on a fixed
`(device, precision, OpenVINO version)`.


In [ ]:
# Replay: reload + reseed from scratch, then compare call-by-call.
rep_model, _ = load(seed=SEED)
replay = run(rep_model, build_fixed_inputs(rep_model), out_name, CALLS)

all_match = True
for k in range(CALLS):
    diff = max_abs(recorded[k], replay[k])
    all_match &= diff == 0

## Takeaways for the golden-action implementation

- **Seed at the IR level.** Rewrite the `RandomUniform` op(s) with non-zero
  `global_seed`/`op_seed` before `compile_model`. There is no runtime knob and no
  `noise` input to set.
- **Record and replay an ordered sequence**, re-initializing the seeded model per
  reference — a single call is not idempotent (the sequence advances).
- **Exact equality works** on a fixed `(device, precision, OpenVINO version)`.
  Across the GPU matrix (PTL/B580/B60/B70) kernel numerics differ, so keep a
  per-device reference (or a tolerance) there.
